# Proximity Analysis
Python version

# Introduction
In this tutorial, you'll explore several techniques for proximity analysis. In particular, you'll learn how to do such things as:

- measure the distance between points on a map, and
- select all points within some radius of a feature.

In [ ]:

from pyprojroot import here
import folium
from folium import Marker, GeoJson
from folium.plugins import HeatMap
import geopandas as gpd


You'll work with a dataset from the US Environmental Protection Agency (EPA) that tracks releases of toxic chemicals in Philadelphia, Pennsylvania, USA.

In [ ]:
releases = gpd.read_file(here("data/toxic_release_pennsylvania/toxic_release_pennsylvania/toxic_release_pennsylvania.shp")) 
releases.head()

You'll also work with a dataset that contains readings from air quality monitoring stations in the same city.

In [ ]:
stations = gpd.read_file(here("data/PhillyHealth_Air_Monitoring_Stations/PhillyHealth_Air_Monitoring_Stations/PhillyHealth_Air_Monitoring_Stations.shp"))
stations.head()

## Measuring distance
To measure distances between points from two different GeoDataFrames, we first have to make sure that they use the same coordinate reference system (CRS). Thankfully, this is the case here, where both use EPSG 2272.


In [ ]:
print(stations.crs)
print(releases.crs)

We also check the CRS to see which units it uses (meters, feet, or something else). In this case, EPSG 2272 has units of feet. (If you like, you can check this [here](https://epsg.io/2272).)

It's relatively straightforward to compute distances in GeoPandas. The code cell below calculates the distance (in feet) between a relatively recent release incident in recent_release and every station in the stations GeoDataFrame.

In [ ]:
# Select one release incident in particular
recent_release = releases.iloc[360]

# Measure distance from release to each station
distances = stations.geometry.distance(recent_release.geometry)
distances


Using the calculated distances, we can obtain statistics like the mean distance to each station.

In [ ]:
print('Mean distance to monitoring stations: {} feet'.format(distances.mean()))

Or, we can get the closest monitoring station.

In [ ]:
print('Closest monitoring station ({} feet):'.format(distances.min()))
print(stations.iloc[distances.idxmin()][["ADDRESS", "LATITUDE", "LONGITUDE"]])

## Creating a buffer
If we want to understand all points on a map that are some radius away from a point, the simplest way is to create a buffer.

The code cell below creates a GeoSeries two_mile_buffer containing 12 different Polygon objects. Each polygon is a buffer of 2 miles (or, 2*5280 feet) around a different air monitoring station.



In [ ]:
two_mile_buffer = stations.geometry.buffer(2*5280)
two_mile_buffer.head()

We use folium.GeoJson() to plot each polygon on a map. Note that since folium requires coordinates in latitude and longitude, we have to convert the CRS to EPSG 4326 before plotting.

In [ ]:
# Create map with release incidents and monitoring stations
m = folium.Map(location=[39.9526,-75.1652], zoom_start=11)
HeatMap(data=releases[['LATITUDE', 'LONGITUDE']], radius=15).add_to(m)
for idx, row in stations.iterrows():
    Marker([row['LATITUDE'], row['LONGITUDE']]).add_to(m)
    
# Plot each polygon on the map
GeoJson(two_mile_buffer.to_crs(epsg=4326)).add_to(m)

# Show the map
m

Now, to test if a toxic release occurred within 2 miles of any monitoring station, we could run 12 different tests for each polygon (to check individually if it contains the point).

But a more efficient way is to first collapse all of the polygons into a MultiPolygon object. We do this with the union_all() method.

In [ ]:
# Turn group of polygons into single multipolygon
my_union = two_mile_buffer.geometry.union_all()
print('Type:', type(my_union))

# Show the MultiPolygon object
my_union

We use the contains() method to check if the multipolygon contains a point. We'll use the release incident from earlier in the tutorial, which we know is roughly 3781 feet to the closest monitoring station.

In [ ]:
# The closest station is less than two miles away
my_union.contains(releases.iloc[360].geometry)

But not all releases occured within two miles of an air monitoring station!

In [ ]:
# The closest station is more than two miles away
my_union.contains(releases.iloc[358].geometry)

In [ ]:
from pyprojroot import here
import geopandas as gpd
from shapely.geometry import Point

import folium
from folium import Marker
from folium.plugins import HeatMap

You are part of a crisis response team, and you want to identify how hospitals have been responding to crash collisions in New York City.

# Exercises
1) Visualize the collision data.
Run the code cell below to load a GeoDataFrame collisions tracking major motor vehicle collisions in 2013-2018.

In [ ]:
collisions = gpd.read_file(here("data/NYPD_Motor_Vehicle_Collisions/NYPD_Motor_Vehicle_Collisions/NYPD_Motor_Vehicle_Collisions.shp"))
collisions.head()

In [ ]:
m_1 = folium.Map(location=[40.7, -74], zoom_start=11)

HeatMap(data=collisions[['LATITUDE', 'LONGITUDE']], radius=10).add_to(m_1)
# for idx, row in collisions.iterrows():
#    Marker([row['LATITUDE'], row['LONGITUDE']]).add_to(m_1)

m_1

Load the hospital data

In [ ]:
hospitals = gpd.read_file(here("data/nyu_2451_34494/nyu_2451_34494/nyu_2451_34494.shp"))
hospitals.head()

Use the "latitude" and "longitude" columns to visualize the hospital locations. 

In [ ]:
m_2 = folium.Map(location=[40.7, -74], zoom_start=11)

for idx, row in hospitals.iterrows():
    Marker([row['latitude'], row['longitude']], popup=row['name']).add_to(m_2)

m_2

3) When was the closest hospital more than 10 kilometers away?
Create a DataFrame outside_range containing all rows from collisions with crashes that occurred more than 10 kilometers from the closest hospital.

Note that both hospitals and collisions have EPSG 2263 as the coordinate reference system, and EPSG 2263 has units of meters.

In [ ]:
tenk_buffer = hospitals.geometry.buffer(10*1000)
tenk_buffer_union = tenk_buffer.geometry.union_all()

outside_range = collisions.loc[~collisions["geometry"].apply(lambda x: tenk_buffer_union.contains(x))]

The next code cell calculates the percentage of collisions that occurred more than 10 kilometers away from the closest hospital.

In [ ]:
percentage = round(100*len(outside_range)/len(collisions), 2)
print("Percentage of collisions more than 10 km away from the closest hospital: {}%".format(percentage))

### 4) Make a recommender.

When collisions occur in distant locations, it becomes even more vital that injured persons are transported to the nearest available hospital.

With this in mind, you decide to create a recommender that:
- takes the location of the crash (in EPSG 2263) as input,
- finds the closest hospital (where distance calculations are done in EPSG 2263), and 
- returns the name of the closest hospital. 

In [ ]:

def best_hospital(collision_location): 
    # distances = hospitals.geometry.distance(collision_location)
    # name = hospitals.iloc[distances.idxmin()]["name"]
    idx_min = hospitals.geometry.distance(collision_location).idxmin()
    my_hospital = hospitals.iloc[idx_min]
    name = my_hospital["name"]
    return name

best_hospital(outside_range.geometry.iloc[0])


### 5) Which hospital is under the highest demand?

Considering only collisions in the `outside_range` DataFrame, which hospital is most recommended?  

Your answer should be a Python string that exactly matches the name of the hospital returned by the function you created in **4)**.

In [ ]:
outside_range.shape

In [ ]:
best_hospital_series = outside_range.geometry.apply(best_hospital)

In [ ]:
best_hospital_series.value_counts().idxmax()

### 6) Where should the city construct new hospitals?

Run the next code cell (without changes) to visualize hospital locations, in addition to collisions that occurred more than 10 kilometers away from the closest hospital. 

Click anywhere on the map to see a pop-up with the corresponding location in latitude and longitude.

The city of New York reaches out to you for help with deciding locations for two brand new hospitals.  They specifically want your help with identifying locations to bring the calculated percentage from step **3)** to less than ten percent.  Using the map (and without worrying about zoning laws or what potential buildings would have to be removed in order to build the hospitals), can you identify two locations that would help the city accomplish this goal?  

Put the proposed latitude and longitude for hospital 1 in `lat_1` and `long_1`, respectively.  (Likewise for hospital 2.)

Then, run the rest of the cell as-is to see the effect of the new hospitals.  Your answer will be marked correct, if the two new hospitals bring the percentage to less than ten percent.

In [ ]:
m_6 = folium.Map(location=[40.7, -74], zoom_start=11) 

coverage = gpd.GeoDataFrame(geometry=hospitals.geometry).buffer(10000)
folium.GeoJson(coverage.geometry.to_crs(epsg=4326)).add_to(m_6)
HeatMap(data=outside_range[['LATITUDE', 'LONGITUDE']], radius=9).add_to(m_6)
folium.LatLngPopup().add_to(m_6)

m_6

New coverage map with new hospital long/lat

In [ ]:
geometry = [
    Point(-73.7574, 40.679), 
    Point(-73.8631, 40.6753)]

new_hosp_gdf = gpd.GeoDataFrame(geometry=[Point(-73.7574, 40.679), Point(-73.8631, 40.6753)], crs="EPSG:4326")
new_hosp_coverage = new_hosp_gdf.to_crs(epsg=2263).buffer(10000)

In [ ]:
new_my_union = new_hosp_coverage.geometry.union_all()
new_outside_range = outside_range.loc[~outside_range["geometry"].apply(lambda x: new_my_union.contains(x))]
new_percentage = round(100*len(new_outside_range)/len(collisions), 2)
print("(NEW) Percentage of collisions more than 10 km away from the closest hospital: {}%".format(new_percentage))

In [ ]:
m_6_1 = folium.Map(location=[40.7, -74], zoom_start=11) 

coverage = gpd.GeoDataFrame(geometry=hospitals.geometry).buffer(10000)
folium.GeoJson(coverage.geometry.to_crs(epsg=4326)).add_to(m_6_1)
HeatMap(data=new_outside_range[['LATITUDE', 'LONGITUDE']], radius=9).add_to(m_6_1)
folium.GeoJson(new_hosp_coverage.geometry.to_crs(epsg=4326)).add_to(m_6_1)
folium.LatLngPopup().add_to(m_6_1)

m_6_1

